In [2]:
# =============================================================================
# MetaDiv Subset Builder
# =============================================================================
# Independent Jupyter Notebook module for creating analytical subsets from
# MetaDiv Builder FINAL_DB outputs.
#
# Modes:
# - headers  : subset by SPPN/OTU IDs from a TXT list
# - sites    : subset by selected sample/site columns from a TXT list
# - taxonomy : subset by taxonomic rank and value
#
# Author:
# Bernardo Águila, UNAM
# =============================================================================

from pathlib import Path
import pandas as pd
import re


# =============================================================================
# 1. USER CONFIGURATION
# =============================================================================

PROJECT_DIR = Path(r"C:\Users\berna\Desktop\META DIV BUILDER V1.6")

MODE = "ITS"  # ITS or 16S

SUBSET_MODE = "headers"  # headers | sites | taxonomy

SUBSET_NAME = "subset_Rusulas_species_p1_sppn08"

# If None, the script will try to auto-detect the FINAL_DB file.
FINAL_DB_FILE =  r"C:\Users\berna\Desktop\META DIV BUILDER V1.6\output\ITS\FINAL_DB\Final_Database_species_only_only_fungi_p1.csv"

# Example manual path:
# FINAL_DB_FILE = PROJECT_DIR / "output" / "ITS" / "FINAL_DB" / "Final_Database_species_only_only_fungi_p1.csv"

QUERY_LISTS_DIR = PROJECT_DIR / "subsets" / "query_lists"

HEADERS_FILE = QUERY_LISTS_DIR / "headers.txt"
SITES_FILE = QUERY_LISTS_DIR / "sites.txt"

# Taxonomy mode settings
TAXONOMIC_RANK = "family"
TAXONOMIC_FILTER = "Amanitaceae"
TAX_FILTER_MODE = "loose"  # exact | loose | contains

# Filtering behavior
DROP_ZERO_OTUS = True
DROP_ZERO_SITES = True
RECALCULATE_TOTAL_ABUNDANCE = True

# Optional site rename
REMOVE_SITE_PREFIX = "Sitio_"  # use None to disable


# =============================================================================
# 2. OUTPUT DIRECTORIES
# =============================================================================

SUBSETS_DIR = PROJECT_DIR / "subsets"

MODE_OUTPUT_DIR = SUBSETS_DIR / f"subset_by_{SUBSET_MODE}"

OUTPUT_DIR = MODE_OUTPUT_DIR / SUBSET_NAME

FOR_R_OUTPUT_DIR = OUTPUT_DIR / "For_R"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FOR_R_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUERY_LISTS_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 3. CONSTANTS
# =============================================================================

TAX_RANKS = [
    "domain",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species"
]

PVALUE_COLS = [f"{r}_pvalue" for r in TAX_RANKS]

CORE_ID_COLS = [
    "SPPN",
    "OTU_XX",
    "Original_OTUID",
    "Original_ID",
    "OTU_Number"
]

KNOWN_NON_SITE_COLS = set(
    CORE_ID_COLS
    + TAX_RANKS
    + PVALUE_COLS
    + [
        "sequence",
        "sequence_lenght",
        "sequence_length",
        "Total_Abundance",
        "sintax_taxonomy",
        "primary_lifestyle",
        "Secondary_lifestyle",
        "secondary_lifestyle",
        "functional_group",
        "FAPROTAX_functions",
        "FAPROTAX_groups"
    ]
)


# =============================================================================
# 4. HELPER FUNCTIONS
# =============================================================================

def find_final_db_file():
    final_db_dir = PROJECT_DIR / "output" / MODE / "FINAL_DB"
    
    if FINAL_DB_FILE is not None:
        path = Path(FINAL_DB_FILE)
        if not path.exists():
            raise FileNotFoundError(f"FINAL_DB_FILE not found: {path}")
        return path
    
    candidates = sorted(final_db_dir.glob("Final_Database_*.csv"))
    
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No Final_Database_*.csv file found in: {final_db_dir}"
        )
    
    if len(candidates) > 1:
        print("Multiple FINAL_DB files detected:")
        for i, c in enumerate(candidates, start=1):
            print(f"{i}. {c}")
        raise ValueError(
            "Please set FINAL_DB_FILE manually to avoid ambiguity."
        )
    
    return candidates[0]


def normalize_text(x):
    return str(x).strip().lower()


def clean_header_id(x):
    x = str(x).strip()
    if x.startswith(">"):
        x = x[1:].strip()
    return x.split()[0]


def load_txt_list(file_path):
    file_path = Path(file_path)
    
    if not file_path.exists():
        raise FileNotFoundError(f"List file not found: {file_path}")
    
    with open(file_path, "r", encoding="utf-8") as f:
        values = [
            line.strip()
            for line in f
            if line.strip()
        ]
    
    if not values:
        raise ValueError(f"List file is empty: {file_path}")
    
    return values


def detect_id_column(df):
    for col in CORE_ID_COLS:
        if col in df.columns:
            return col
    raise ValueError("No valid ID column found. Expected SPPN or OTU_XX.")


def detect_site_columns(df):
    site_cols = []
    
    for col in df.columns:
        col_str = str(col)
        col_lower = col_str.lower()
        
        if col in KNOWN_NON_SITE_COLS:
            continue
        
        if col_lower.startswith("sequence"):
            continue
        
        if col_lower.endswith("_pvalue"):
            continue
        
        numeric_values = pd.to_numeric(
            df[col],
            errors="coerce"
        )
        
        non_na_fraction = numeric_values.notna().mean()
        
        if non_na_fraction > 0.80:
            site_cols.append(col)
    
    return site_cols


def rename_sites(cols, prefix):
    if not prefix:
        return cols
    
    renamed = []
    
    for col in cols:
        if isinstance(col, str) and col.startswith(prefix):
            renamed.append(col[len(prefix):])
        else:
            renamed.append(col)
    
    return renamed


def make_safe_sample_names(site_cols):
    safe_names = []
    used = {}
    
    for original in site_cols:
        name = str(original).strip()
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
        safe = re.sub(r"_+", "_", safe).strip("_")
        
        if not safe:
            safe = "Sample"
        
        if safe[0].isdigit():
            safe = f"S_{safe}"
        
        base = safe
        
        if base in used:
            used[base] += 1
            safe = f"{base}_{used[base]}"
        else:
            used[base] = 1
        
        safe_names.append(safe)
    
    sample_map = pd.DataFrame({
        "Original_SampleID": [str(c) for c in site_cols],
        "Safe_SampleID": safe_names
    })
    
    return safe_names, sample_map


def apply_taxonomic_filter(df, rank, filt, mode):
    if rank not in df.columns:
        raise ValueError(f"Taxonomic rank not found: {rank}")
    
    if isinstance(filt, (str, int, float)):
        filters = [str(filt)]
    else:
        filters = [str(x) for x in filt]
    
    col = df[rank].astype(str)
    
    if mode == "exact":
        mask = col.isin(filters)
    
    elif mode == "loose":
        filters_norm = {normalize_text(x) for x in filters}
        mask = col.apply(normalize_text).isin(filters_norm)
    
    elif mode == "contains":
        pattern = "|".join(re.escape(x) for x in filters)
        mask = col.str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
    
    else:
        raise ValueError("TAX_FILTER_MODE must be exact, loose, or contains.")
    
    out = df[mask].copy()
    
    print(f"Taxonomic filter applied: {rank} | {mode} | {filters}")
    print(f"Rows before: {len(df)}")
    print(f"Rows after : {len(out)}")
    
    return out


def export_fasta(df, id_col, out_fasta):
    if id_col not in df.columns or "sequence" not in df.columns:
        print("FASTA export skipped because ID or sequence column is missing.")
        return 0
    
    n = 0
    
    with open(out_fasta, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            seq_id = str(row[id_col]).strip()
            seq = str(row["sequence"]).strip()
            
            if (
                seq_id
                and seq
                and seq_id.lower() != "nan"
                and seq.lower() != "nan"
            ):
                f.write(f">{seq_id}\n{seq}\n")
                n += 1
    
    return n


def export_for_r_files(subset, id_col, site_cols_final):
    safe_site_names, sample_map = make_safe_sample_names(site_cols_final)
    rename_map = dict(zip(site_cols_final, safe_site_names))
    
    # Abundance table
    if site_cols_final:
        abundance = subset[[id_col] + site_cols_final].copy()
        
        for col in site_cols_final:
            abundance[col] = pd.to_numeric(
                abundance[col],
                errors="coerce"
            ).fillna(0).astype("int64")
        
        abundance = abundance.rename(columns=rename_map)
        abundance = abundance.drop_duplicates(subset=[id_col], keep="first")
        abundance = abundance.set_index(id_col)
    else:
        abundance = subset[[id_col]].drop_duplicates(subset=[id_col], keep="first")
        abundance = abundance.set_index(id_col)
    
    abundance.to_csv(
        FOR_R_OUTPUT_DIR / "abundance_table.csv",
        encoding="utf-8",
        na_rep="0"
    )
    
    # Taxonomy table
    tax_cols = [c for c in TAX_RANKS if c in subset.columns]
    pvalue_cols = [c for c in PVALUE_COLS if c in subset.columns]
    
    ecological_cols = [
        c for c in subset.columns
        if c not in site_cols_final
        and c not in tax_cols
        and c not in pvalue_cols
        and c not in [id_col]
        and c in [
            "primary_lifestyle",
            "Secondary_lifestyle",
            "secondary_lifestyle",
            "functional_group",
            "FAPROTAX_functions",
            "FAPROTAX_groups"
        ]
    ]
    
    taxonomy_cols = [id_col] + tax_cols + pvalue_cols + ecological_cols
    
    taxonomy = subset[taxonomy_cols].drop_duplicates(
        subset=[id_col],
        keep="first"
    )
    
    taxonomy.to_csv(
        FOR_R_OUTPUT_DIR / "taxonomy_table.csv",
        index=False,
        encoding="utf-8"
    )
    
    taxonomy.to_csv(
        FOR_R_OUTPUT_DIR / "taxonomy.csv",
        index=False,
        encoding="utf-8"
    )
    
    # Metadata template
    metadata = pd.DataFrame({
        "SampleID": safe_site_names,
        "Group": "All_samples",
        "Site": safe_site_names
    })
    
    metadata.to_csv(
        FOR_R_OUTPUT_DIR / "metadata_template.csv",
        index=False,
        encoding="utf-8"
    )
    
    metadata.to_csv(
        FOR_R_OUTPUT_DIR / "sample_metadata_to_fill.csv",
        index=False,
        encoding="utf-8"
    )
    
    sample_map.to_csv(
        FOR_R_OUTPUT_DIR / "sample_name_map.csv",
        index=False,
        encoding="utf-8"
    )
    
    # FASTA
    fasta_count = export_fasta(
        subset,
        id_col,
        FOR_R_OUTPUT_DIR / "sequences.fasta"
    )
    
    # Info file
    with open(FOR_R_OUTPUT_DIR / "FOR_R_INFO.txt", "w", encoding="utf-8") as f:
        f.write("MetaDiv Subset Builder - For_R export\n")
        f.write("=====================================\n\n")
        f.write(f"Subset name: {SUBSET_NAME}\n")
        f.write(f"Subset mode: {SUBSET_MODE}\n")
        f.write(f"Taxa/rows: {len(subset)}\n")
        f.write(f"Sites/samples: {len(site_cols_final)}\n")
        f.write(f"FASTA sequences: {fasta_count}\n")
    
    return {
        "for_r_abundance": FOR_R_OUTPUT_DIR / "abundance_table.csv",
        "for_r_taxonomy": FOR_R_OUTPUT_DIR / "taxonomy_table.csv",
        "for_r_fasta": FOR_R_OUTPUT_DIR / "sequences.fasta",
        "for_r_metadata": FOR_R_OUTPUT_DIR / "metadata_template.csv"
    }


# =============================================================================
# 5. LOAD FINAL DB
# =============================================================================

INPUT_FILE = find_final_db_file()

df = pd.read_csv(INPUT_FILE, low_memory=False)

id_col = detect_id_column(df)

print("Input FINAL_DB:")
print(INPUT_FILE)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("ID column:", id_col)


# =============================================================================
# 6. DETECT ORIGINAL SITE COLUMNS
# =============================================================================

all_site_cols = detect_site_columns(df)

print("Detected site columns:", len(all_site_cols))


# =============================================================================
# 7. APPLY SUBSET MODE
# =============================================================================

if SUBSET_MODE == "headers":
    
    header_ids = [
        clean_header_id(x)
        for x in load_txt_list(HEADERS_FILE)
    ]
    
    subset = df[df[id_col].astype(str).isin(header_ids)].copy()
    
    selected_site_cols = all_site_cols.copy()
    
    print("Header IDs requested:", len(header_ids))
    print("Rows retained:", len(subset))


elif SUBSET_MODE == "sites":
    
    selected_sites = load_txt_list(SITES_FILE)
    
    missing_sites = [
        s for s in selected_sites
        if s not in df.columns
    ]
    
    if missing_sites:
        print("Missing sites:")
        for s in missing_sites[:30]:
            print("-", s)
        raise ValueError("Some requested sites were not found.")
    
    non_site_cols = [
        c for c in df.columns
        if c not in all_site_cols
    ]
    
    selected_site_cols = selected_sites.copy()
    
    subset = df[non_site_cols + selected_site_cols].copy()
    
    print("Selected sites:", len(selected_site_cols))


elif SUBSET_MODE == "taxonomy":
    
    subset = apply_taxonomic_filter(
        df,
        TAXONOMIC_RANK,
        TAXONOMIC_FILTER,
        TAX_FILTER_MODE
    )
    
    selected_site_cols = [
        c for c in all_site_cols
        if c in subset.columns
    ]


else:
    raise ValueError("SUBSET_MODE must be headers, sites, or taxonomy.")


# =============================================================================
# 8. NUMERIC SITE COLUMNS AND PRUNING
# =============================================================================

if selected_site_cols:
    
    subset[selected_site_cols] = subset[selected_site_cols].apply(
        pd.to_numeric,
        errors="coerce"
    ).fillna(0)
    
    if DROP_ZERO_OTUS:
        before = len(subset)
        subset = subset[
            subset[selected_site_cols].sum(axis=1) > 0
        ].copy()
        after = len(subset)
        print(f"Zero-OTU pruning: {before} -> {after}")
    
    if DROP_ZERO_SITES:
        site_sums = subset[selected_site_cols].sum(axis=0)
        zero_sites = site_sums[site_sums <= 0].index.tolist()
        
        if zero_sites:
            subset = subset.drop(columns=zero_sites, errors="ignore")
            selected_site_cols = [
                c for c in selected_site_cols
                if c not in zero_sites
            ]
            print("Zero-site columns removed:", len(zero_sites))
        else:
            print("No zero-site columns removed.")
    
    if RECALCULATE_TOTAL_ABUNDANCE:
        subset["Total_Abundance"] = subset[selected_site_cols].sum(axis=1)
        print("Total_Abundance recalculated.")

else:
    print("No site columns available for pruning.")


# =============================================================================
# 9. OPTIONAL SITE RENAME
# =============================================================================

site_cols_final = selected_site_cols.copy()

if REMOVE_SITE_PREFIX and site_cols_final:
    
    renamed_sites = rename_sites(site_cols_final, REMOVE_SITE_PREFIX)
    
    rename_map = {
        old: new
        for old, new in zip(site_cols_final, renamed_sites)
        if old != new
    }
    
    if rename_map:
        subset = subset.rename(columns=rename_map)
        print(f"Removed site prefix: {REMOVE_SITE_PREFIX}")
    
    site_cols_final = renamed_sites


# =============================================================================
# 10. EXPORT MAIN SUBSET FILES
# =============================================================================

out_final_db = OUTPUT_DIR / f"{SUBSET_NAME}_final_db.csv"
out_fasta = OUTPUT_DIR / f"{SUBSET_NAME}_sequences.fasta"
out_abundance = OUTPUT_DIR / f"{SUBSET_NAME}_abundance_table.csv"
out_taxonomy = OUTPUT_DIR / f"{SUBSET_NAME}_taxonomy_table.csv"
out_summary = OUTPUT_DIR / f"{SUBSET_NAME}_summary.txt"

subset.to_csv(
    out_final_db,
    index=False,
    encoding="utf-8"
)

fasta_count = export_fasta(
    subset,
    id_col,
    out_fasta
)

if site_cols_final:
    abundance_export = subset[[id_col] + site_cols_final].copy()
    abundance_export.to_csv(
        out_abundance,
        index=False,
        encoding="utf-8"
    )

tax_export_cols = (
    [id_col]
    + [c for c in TAX_RANKS if c in subset.columns]
    + [c for c in PVALUE_COLS if c in subset.columns]
)

tax_export_cols = list(dict.fromkeys(tax_export_cols))

subset[tax_export_cols].to_csv(
    out_taxonomy,
    index=False,
    encoding="utf-8"
)


# =============================================================================
# 11. EXPORT For_R PACKAGE
# =============================================================================

for_r_paths = export_for_r_files(
    subset=subset,
    id_col=id_col,
    site_cols_final=site_cols_final
)


# =============================================================================
# 12. WRITE SUMMARY
# =============================================================================

summary_lines = [
    "MetaDiv Subset Builder summary",
    "==============================",
    "",
    f"Project directory: {PROJECT_DIR}",
    f"Input FINAL_DB: {INPUT_FILE}",
    f"Subset mode: {SUBSET_MODE}",
    f"Subset name: {SUBSET_NAME}",
    "",
    f"Original rows: {len(df)}",
    f"Subset rows: {len(subset)}",
    f"Detected original site columns: {len(all_site_cols)}",
    f"Retained site columns: {len(site_cols_final)}",
    f"FASTA sequences exported: {fasta_count}",
    "",
    "Main outputs:",
    f"- {out_final_db}",
    f"- {out_abundance}",
    f"- {out_taxonomy}",
    f"- {out_fasta}",
    "",
    "For_R outputs:",
    f"- {for_r_paths['for_r_abundance']}",
    f"- {for_r_paths['for_r_taxonomy']}",
    f"- {for_r_paths['for_r_fasta']}",
    f"- {for_r_paths['for_r_metadata']}",
]

if SUBSET_MODE == "taxonomy":
    summary_lines += [
        "",
        "Taxonomy filter:",
        f"- Rank: {TAXONOMIC_RANK}",
        f"- Filter: {TAXONOMIC_FILTER}",
        f"- Mode: {TAX_FILTER_MODE}"
    ]

with open(out_summary, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("\nDONE")
print("Subset folder:")
print(OUTPUT_DIR)
print("\nFor_R folder:")
print(FOR_R_OUTPUT_DIR)

Input FINAL_DB:
C:\Users\berna\Desktop\META DIV BUILDER V1.6\output\ITS\FINAL_DB\Final_Database_species_only_only_fungi_p1.csv
Rows: 114928
Columns: 348
ID column: SPPN
Detected site columns: 326
Header IDs requested: 25
Rows retained: 25
Zero-OTU pruning: 25 -> 25
Zero-site columns removed: 306
Total_Abundance recalculated.

DONE
Subset folder:
C:\Users\berna\Desktop\META DIV BUILDER V1.6\subsets\subset_by_headers\subset_Rusulas_species_p1_sppn08

For_R folder:
C:\Users\berna\Desktop\META DIV BUILDER V1.6\subsets\subset_by_headers\subset_Rusulas_species_p1_sppn08\For_R
